<a href="https://colab.research.google.com/github/4GTTN/Colab/blob/main/todo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее задание №16 - Классификация текстов

## Импорт библиотек

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import os
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Embedding, SpatialDropout1D, Flatten
from tensorflow.keras.layers import Conv1D, MaxPooling1D, GlobalMaxPooling1D, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
!rm -R /content/texts 2>/dev/null
!unzip -q '/content/drive/My Drive/Базы/Тексты писателей.zip' -d /content/texts

def readText(fileName):
    f = open(fileName, 'r', encoding='utf-8')
    text = f.read()
    text = text.replace("\n", " ")
    return text

className = ["О. Генри", "Стругацкие", "Булгаков", "Саймак", "Фрай", "Брэдберри"]
nClasses = len(className)

trainText = []
testText = []

for i in className:
    for j in os.listdir('texts/'):
        if i in j:
            if 'Обучающая' in j:
                trainText.append(readText('texts/' + j))
                print(j, 'добавлен в обучающую выборку')
            if 'Тестовая' in j:
                testText.append(readText('texts/' + j))
                print(j, 'добавлен в тестовую выборку')
    print()

print(f"Количество классов: {nClasses}")
print(f"Размер обучающей выборки: {len(trainText)}")
print(f"Размер тестовой выборки: {len(testText)}")


(О. Генри) Тестовая_20 вместе.txt добавлен в тестовую выборку
(О. Генри) Обучающая_50 вместе.txt добавлен в обучающую выборку

(Стругацкие) Обучающая_5 вместе.txt добавлен в обучающую выборку
(Стругацкие) Тестовая_2 вместе.txt добавлен в тестовую выборку

(Булгаков) Обучающая_5 вместе.txt добавлен в обучающую выборку
(Булгаков) Тестовая_2 вместе.txt добавлен в тестовую выборку

(Клиффорд_Саймак) Тестовая_2 вместе.txt добавлен в тестовую выборку
(Клиффорд_Саймак) Обучающая_5 вместе.txt добавлен в обучающую выборку

(Макс Фрай) Тестовая_2 вместе.txt добавлен в тестовую выборку
(Макс Фрай) Обучающая_5 вместе.txt добавлен в обучающую выборку

(Рэй Брэдберри) Обучающая_22 вместе.txt добавлен в обучающую выборку
(Рэй Брэдберри) Тестовая_8 вместе.txt добавлен в тестовую выборку

Количество классов: 6
Размер обучающей выборки: 6
Размер тестовой выборки: 6


In [8]:
def getSetFromIndexes(wordIndexes, xLen, step):
    xSample = []
    wordsLen = len(wordIndexes)
    index = 0
    while (index + xLen <= wordsLen):
        xSample.append(wordIndexes[index:index+xLen])
        index += step
    return xSample

def createSetsMultiClasses(wordIndexes, xLen, step):
    nClasses = len(wordIndexes)
    classesXSamples = []
    for wI in wordIndexes:
        classesXSamples.append(getSetFromIndexes(wI, xLen, step))

    xSamples = []
    ySamples = []

    for t in range(nClasses):
        xT = classesXSamples[t]
        for i in range(len(xT)):
            xSamples.append(xT[i])
            ySamples.append(to_categorical(t, nClasses))

    xSamples = np.array(xSamples)
    ySamples = np.array(ySamples)

    return (xSamples, ySamples)

In [9]:
def plot_training_history(history, title):
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Обучающая')
    plt.plot(history.history['val_accuracy'], label='Проверочная')
    plt.title(f'{title} - Точность')
    plt.xlabel('Эпоха')
    plt.ylabel('Точность')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Обучающая')
    plt.plot(history.history['val_loss'], label='Проверочная')
    plt.title(f'{title} - Потери')
    plt.xlabel('Эпоха')
    plt.ylabel('Потери')
    plt.legend()

    plt.tight_layout()
    plt.show()

# LIGHT Вариант 1

## Задание 1: Bag of Words при разных maxWordsCount

In [10]:
def train_bow_model(max_words, x_train, y_train, x_val, y_val, epochs=20):
    x_train_trim = x_train[:, :max_words]
    x_val_trim = x_val[:, :max_words]

    model = Sequential()
    model.add(Dense(200, input_dim=max_words, activation="relu"))
    model.add(Dropout(0.25))
    model.add(BatchNormalization())
    model.add(Dense(nClasses, activation='sigmoid'))

    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    start_time = time.time()
    history = model.fit(x_train_trim, y_train,
                        epochs=epochs,
                        batch_size=128,
                        validation_data=(x_val_trim, y_val),
                        verbose=0)
    train_time = time.time() - start_time

    final_val_acc = history.history['val_accuracy'][-1]
    print(f"maxWordsCount = {max_words:5d} | Время: {train_time:.2f} сек | Точность: {final_val_acc:.4f}")

    return model, history, final_val_acc

# Подготовка данных для заданий 1 и 2
maxWordsCount_base = 20000
tokenizer_base = Tokenizer(num_words=maxWordsCount_base,
                           filters='!\"#$%&()*+,-–—./…:;<=>?@[\\\\]^_`{|}~«»\\t\\n\\xa0',
                           lower=True, split=' ', oov_token='unknown', char_level=False)

tokenizer_base.fit_on_texts(trainText)
trainWordIndexes = tokenizer_base.texts_to_sequences(trainText)
testWordIndexes = tokenizer_base.texts_to_sequences(testText)

xLen = 1000
step = 100
xTrain, yTrain = createSetsMultiClasses(trainWordIndexes, xLen, step)
xTest, yTest = createSetsMultiClasses(testWordIndexes, xLen, step)

xTrain01 = tokenizer_base.sequences_to_matrix(xTrain.tolist())
xTest01 = tokenizer_base.sequences_to_matrix(xTest.tolist())

print(f"Размер обучающей выборки: {xTrain01.shape}")
print(f"Размер тестовой выборки: {xTest01.shape}")

print("\n" + "="*60)
print("Задание 1: Bag of Words при разных maxWordsCount")
print("="*60)

bow_results = {}
max_words_list = [100, 1000, 10000, 20000, 50000]

for max_w in max_words_list:
    if max_w <= maxWordsCount_base:
        _, _, acc = train_bow_model(max_w, xTrain01, yTrain, xTest01, yTest, epochs=15)
        bow_results[max_w] = acc
    else:
        print(f"maxWordsCount = {max_w} превышает базовый размер {maxWordsCount_base}, пропускаем")

print("\n--- Итоговые результаты Bag of Words ---")
for m, acc in bow_results.items():
    print(f"maxWordsCount = {m:5d}: Точность = {acc:.4f}")

Размер обучающей выборки: (17765, 20000)
Размер тестовой выборки: (6748, 20000)

Задание 1: Bag of Words при разных maxWordsCount
maxWordsCount =   100 | Время: 12.88 сек | Точность: 0.6494
maxWordsCount =  1000 | Время: 25.51 сек | Точность: 0.8847
maxWordsCount = 10000 | Время: 135.44 сек | Точность: 0.9063
maxWordsCount = 20000 | Время: 298.22 сек | Точность: 0.9160
maxWordsCount = 50000 превышает базовый размер 20000, пропускаем

--- Итоговые результаты Bag of Words ---
maxWordsCount =   100: Точность = 0.6494
maxWordsCount =  1000: Точность = 0.8847
maxWordsCount = 10000: Точность = 0.9063
maxWordsCount = 20000: Точность = 0.9160


## Задание 2: Bag of Words при разных архитектурах

In [11]:
def train_bow_architecture(x_train, y_train, x_val, y_val, layers_config, activation='relu', epochs=20):
    """
    layers_config: список количества нейронов в каждом слое
    """
    model = Sequential()
    for i, units in enumerate(layers_config):
        if i == 0:
            model.add(Dense(units, input_dim=x_train.shape[1], activation=activation))
        else:
            model.add(Dense(units, activation=activation))
        model.add(Dropout(0.3))
        model.add(BatchNormalization())

    model.add(Dense(nClasses, activation='sigmoid'))

    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    history = model.fit(x_train, y_train,
                        epochs=epochs,
                        batch_size=128,
                        validation_data=(x_val, y_val),
                        verbose=0)
    final_val_acc = history.history['val_accuracy'][-1]
    return model, history, final_val_acc

# Используем данные с maxWordsCount=20000
xTrain_bow = xTrain01
xTest_bow = xTest01

print("\n" + "="*60)
print("Задание 2: Bag of Words при разных архитектурах (maxWordsCount=20000)")
print("="*60)

arch_results = {}

# 1. Исходная архитектура (3 слоя по 200 нейронов)
_, _, acc = train_bow_architecture(xTrain_bow, yTrain, xTest_bow, yTest,
                                   [200, 200, 200], activation='relu', epochs=15)
arch_results['Baseline (3x200, relu)'] = acc

# 2. Меньше нейронов (3 слоя по 50 нейронов)
_, _, acc = train_bow_architecture(xTrain_bow, yTrain, xTest_bow, yTest,
                                   [50, 50, 50], activation='relu', epochs=15)
arch_results['3x50, relu'] = acc

# 3. Больше нейронов (3 слоя по 500 нейронов)
_, _, acc = train_bow_architecture(xTrain_bow, yTrain, xTest_bow, yTest,
                                   [500, 500, 500], activation='relu', epochs=15)
arch_results['3x500, relu'] = acc

# 4. Меньше слоев (1 слой 200 нейронов)
_, _, acc = train_bow_architecture(xTrain_bow, yTrain, xTest_bow, yTest,
                                   [200], activation='relu', epochs=15)
arch_results['1x200, relu'] = acc

# 5. Больше слоев (5 слоев по 200 нейронов)
_, _, acc = train_bow_architecture(xTrain_bow, yTrain, xTest_bow, yTest,
                                   [200, 200, 200, 200, 200], activation='relu', epochs=15)
arch_results['5x200, relu'] = acc

# 6. Другая активационная функция (sigmoid)
_, _, acc = train_bow_architecture(xTrain_bow, yTrain, xTest_bow, yTest,
                                   [200, 200, 200], activation='sigmoid', epochs=15)
arch_results['3x200, sigmoid'] = acc

# 7. Другая активационная функция (tanh)
_, _, acc = train_bow_architecture(xTrain_bow, yTrain, xTest_bow, yTest,
                                   [200, 200, 200], activation='tanh', epochs=15)
arch_results['3x200, tanh'] = acc

print("\n--- Итоговые результаты архитектур ---")
for arch, acc in arch_results.items():
    print(f"{arch:25s}: Точность = {acc:.4f}")


Задание 2: Bag of Words при разных архитектурах (maxWordsCount=20000)

--- Итоговые результаты архитектур ---
Baseline (3x200, relu)   : Точность = 0.9135
3x50, relu               : Точность = 0.8797
3x500, relu              : Точность = 0.8924
1x200, relu              : Точность = 0.9083
5x200, relu              : Точность = 0.9068
3x200, sigmoid           : Точность = 0.9201
3x200, tanh              : Точность = 0.9157


## Задание 3: Embedding + Dense при разных размерах пространства

In [ ]:
maxWordsCount_embed = 50000
tokenizer_embed = Tokenizer(num_words=maxWordsCount_embed,
                            filters='!\"#$%&()*+,-–—./…:;<=>?@[\\\\]^_`{|}~«»\\t\\n\\xa0',
                            lower=True, split=' ', oov_token='unknown', char_level=False)
tokenizer_embed.fit_on_texts(trainText)

trainWordIndexes_embed = tokenizer_embed.texts_to_sequences(trainText)
testWordIndexes_embed = tokenizer_embed.texts_to_sequences(testText)

# Создаем выборку для Embedding
xTrain_embed, yTrain_embed = createSetsMultiClasses(trainWordIndexes_embed, xLen, step)
xTest_embed, yTest_embed = createSetsMultiClasses(testWordIndexes_embed, xLen, step)

print(f"Размер выборки для Embedding: xTrain_embed shape = {xTrain_embed.shape}")

def train_embedding_model(embedding_dim, x_train, y_train, x_val, y_val, epochs=15):
    model = Sequential()
    model.add(Embedding(maxWordsCount_embed, embedding_dim, input_length=xLen))
    model.add(SpatialDropout1D(0.2))
    model.add(Flatten())
    model.add(BatchNormalization())
    model.add(Dense(200, activation="relu"))
    model.add(Dropout(0.2))
    model.add(BatchNormalization())
    model.add(Dense(nClasses, activation='sigmoid'))

    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    history = model.fit(x_train, y_train,
                        epochs=epochs,
                        batch_size=128,
                        validation_data=(x_val, y_val),
                        verbose=0)
    final_val_acc = history.history['val_accuracy'][-1]
    return model, history, final_val_acc

print("\n" + "="*60)
print("Задание 3: Embedding при разных размерах пространства (maxWordsCount=50000)")
print("="*60)

embedding_results = {}
embedding_dims = [10, 50, 200]

for dim in embedding_dims:
    _, _, acc = train_embedding_model(dim, xTrain_embed, yTrain_embed, xTest_embed, yTest_embed, epochs=12)
    embedding_results[dim] = acc
    print(f"Размерность эмбеддинга: {dim:3d}, Точность: {acc:.4f}")

print("\n--- Итоговые результаты для Embedding ---")
for dim, acc in embedding_results.items():
    print(f"Embedding dim = {dim:3d}: Точность = {acc:.4f}")

Размер выборки для Embedding: xTrain_embed shape = (17765, 1000)

Задание 3: Embedding при разных размерах пространства (maxWordsCount=50000)
Размерность эмбеддинга:  10, Точность: 0.6892
Размерность эмбеддинга:  50, Точность: 0.7180
